# Rule-Based ADC Target Ranking and Validation

**v0.1 educational notebook**

## Goal
Build and stress-test an interpretable rule-based ranking baseline. The key question is not only “what ranks highest?” but “does the ranking remain stable when assumptions change?”

## Canonical guide

### Learning objectives
- Convert explicit expert assumptions into a transparent ranking.
- Test sensitivity to weights and thresholds.
- Use ranking stability and Pareto ideas before moving to ML.

### Data mode
Canonical target feature matrix from Notebook 04.

### Inputs
`outputs/adc_target_feature_matrix_v01.csv`

### Canonical outputs
`outputs/adc_rule_based_ranking_v01.csv`, `outputs/adc_weight_sensitivity_v01.csv`, `outputs/adc_threshold_sensitivity_v01.csv`

### Scientific limitation
Weights and thresholds are hypotheses about development priorities, not estimated causal effects or clinical cutoffs.

### Next step
Notebook 06 asks what data-driven models learn relative to this expert baseline.


In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
print('ROOT:', ROOT.resolve())

ROOT: /mnt/data/audit_adc/adc-scientific-ai


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
f=pd.read_csv(OUTPUTS/'adc_target_feature_matrix_v01.csv',index_col=0)
elig=f[f.passes_hard_filters & f.scorable].copy()
benefit_base={'tumor_score':.25,'selectivity_score':.30,'prevalence_score':.25,'surface_score_norm':.20}
risk_base={'normal_protein_risk_norm':.35,'critical_organ_risk':.30,'normal_rna_risk':.20,'heterogeneity_risk':.15}
def rank_table(table,bw,rw):
    r=table.copy(); r['benefit_score']=sum(r[k]*v for k,v in bw.items()); r['risk_score']=sum(r[k]*v for k,v in rw.items()); r['adc_target_score']=r.benefit_score-r.risk_score
    r=r.sort_values('adc_target_score',ascending=False); r['rank']=np.arange(1,len(r)+1); return r
baseline=rank_table(elig,benefit_base,risk_base)
baseline[['rank','benefit_score','risk_score','adc_target_score']].head(10)

,rank,benefit_score,risk_score,adc_target_score
gene_symbol,,,,
GENE_S,1,0.788989,0.123166,0.665823
ERBB2,2,0.979248,0.436797,0.542451
GENE_R,3,0.794185,0.324365,0.469820
GENE_A,4,0.812061,0.425485,0.386576
GENE_G,5,0.825126,0.507450,0.317676
GENE_O,6,0.649254,0.366776,0.282478
GENE_I,7,0.695856,0.550342,0.145515
GENE_Q,8,0.366652,0.336017,0.030635
GENE_C,9,0.629237,0.640295,-0.011058


In [3]:
scenarios={
'baseline':(benefit_base,risk_base),
'selectivity_high':({'tumor_score':.20,'selectivity_score':.40,'prevalence_score':.20,'surface_score_norm':.20},risk_base),
'prevalence_high':({'tumor_score':.20,'selectivity_score':.25,'prevalence_score':.35,'surface_score_norm':.20},risk_base),
'safety_high':(benefit_base,{'normal_protein_risk_norm':.40,'critical_organ_risk':.40,'normal_rna_risk':.10,'heterogeneity_risk':.10})}
ranks=pd.DataFrame({name:rank_table(elig,*cfg)['rank'] for name,cfg in scenarios.items()})
ranks['rank_min']=ranks.min(axis=1); ranks['rank_max']=ranks.max(axis=1); ranks['rank_range']=ranks.rank_max-ranks.rank_min
ranks['stability_score']=1-ranks.rank_range/max(len(ranks)-1,1)
ranks.sort_values(['baseline','rank_range']).head(10)

,baseline,selectivity_high,prevalence_high,safety_high,rank_min,rank_max,rank_range,stability_score
gene_symbol,,,,,,,,
GENE_S,1,1,1,1,1,1,0,1.000000
ERBB2,2,2,2,3,2,3,1,0.916667
GENE_R,3,3,3,2,2,3,1,0.916667
GENE_A,4,4,4,4,4,4,0,1.000000
GENE_G,5,5,5,6,5,6,1,0.916667
GENE_O,6,6,6,5,5,6,1,0.916667
GENE_I,7,7,7,7,7,7,0,1.000000
GENE_Q,8,8,8,8,8,8,0,1.000000
GENE_C,9,12,9,9,9,12,3,0.750000


In [4]:
baseline=baseline.join(ranks[['rank_range','stability_score']])
def pareto_mask(benefit,risk):
    b=np.asarray(benefit); r=np.asarray(risk); keep=np.ones(len(b),dtype=bool)
    for i in range(len(b)):
        dominated=((b>=b[i])&(r<=r[i])&((b>b[i])|(r<r[i]))); dominated[i]=False
        if dominated.any(): keep[i]=False
    return keep
baseline['pareto_candidate']=pareto_mask(baseline.benefit_score,baseline.risk_score)
for name in scenarios:
    if name!='baseline':
        common=baseline.index.intersection(ranks.index); print(name,'Spearman',round(spearmanr(ranks.loc[common,'baseline'],ranks.loc[common,name]).statistic,3))

selectivity_high Spearman 0.962
prevalence_high Spearman 1.0
safety_high Spearman 0.973


In [5]:
baseline.to_csv(OUTPUTS/'adc_rule_based_ranking_v01.csv')
ranks.to_csv(OUTPUTS/'adc_weight_sensitivity_v01.csv')
threshold_summary=[]
for name,s,p,c in [('loose',.5,.10,2),('baseline',.5,.20,2),('strict',.7,.40,1)]:
    n=((f.surface_score>=s)&(f.tumor_prevalence>=p)&(f.critical_organ_max<=c)).sum(); threshold_summary.append((name,n,s,p,c))
pd.DataFrame(threshold_summary,columns=['scenario','n_candidates','surface_min','prevalence_min','critical_max']).to_csv(OUTPUTS/'adc_threshold_sensitivity_v01.csv',index=False)
print('saved ranking and sensitivity outputs')

saved ranking and sensitivity outputs


## Interpretation
Weights and thresholds are hypotheses about the development objective, not biological truths. Stable high-ranking candidates are less sensitive to those assumptions; unstable candidates may represent meaningful trade-offs rather than “bad” targets.